In [2]:
# importing relevent modules
import pandas as pd
import numpy as np

In [3]:
# pulls in data, saved within directory
db = pd.read_csv('./data/share-of-low-carbon-energy-vs-gdp.csv')
db = db[db['Code'].notnull()]

#Removed all data entries with no country code
#Now i can print out all the data , and a quick scroll highlighted final things to change. Our world in Data has created some custom codes for income categories
    #Because of this i need to remove NULL entries AND any entries like (OWID_%). 
    # I also must omit countires with empty GDP data. 
db = db[db['Share of energy from low-carbon sources'].notnull()]
db = db[db['GDP per capita'].notnull()]
db = db[~db['Code'].str.contains('OWID_', na=False)]
# the ~ is the pandas NOT symbol, we are checking each entries code , and if it contains 'OWID_' then it is removed.
db1 = db[db['Year'] >= 1992]
# RQ1 only wants data from 1992 onwards
db2 = db[db['Year'] >= 1990]
# RQ2 wants data from 1990 because there are some extra datapoints that would be helpful
db2.describe()

#	2695 rows! But I dont know which countires are still in our dataset - and if any are missing specific year data. 
# Lets print each country and the number of years covered. 
cdb = db2.groupby(['Entity'])['Year'].count()

#77 countries , each with 35years of data. 

In [4]:
# creating an array of the countries with complete data
country_full = db2["Code"].unique()
x = pd.DataFrame(country_full)

# Now you can save it

x.to_csv("CountryCodes.csv", index=False)

In [5]:
# importing all the policy data
carbon_pricing_df = pd.read_csv("./data/IEA_carbon_pricing.csv")
coal_phasing_df = pd.read_csv("./data/IEA_coal_phase-out.csv")
emissions_standards_df = pd.read_csv("./data/IEA_emissions_standards.csv")
low_carbon_spending_df = pd.read_csv("./data/IEA_low_carbon_spending.csv")
# importing another IEA dataset that has the all of the countries they have policy data on
IEA_all_countries_df = pd.read_csv("./data/IEA_all_power_data.csv")

In [6]:
# putting all the policy data together
policies_df = pd.concat([carbon_pricing_df, coal_phasing_df, emissions_standards_df, low_carbon_spending_df])
policies_df.head(1)

,title,description,status,year,jurisdiction,source,datePromulgated,yearEnded,learnMore,learnMoreLanguage,dateModified,countries,states,technologies,tags,policyType
0,Regional Greenhouse Gas Initiative (RGGI),<div>The Regional Greenhouse Gas Initiative (R...,In force,2008,State/Provincial,JOIN IEA/IRENA Policy and Measures Database,NaN,NaN,http://www.rggi.org/home,NaN,1711642099496,"[{""iso3"":""USA"",""name"":""United States""}]","[{""country"":""US"",""code"":""VT"",""name"":""Vermont""}]",[],[],"[{""topic"":""Power"",""family"":""Market pull"",""name..."


In [7]:
# checking if I need to clean any country data
print(len(policies_df[policies_df['countries'].isnull()]))
# none perfect

0


In [8]:
# creating a dataframe of all the unique countries the IEA has data for
IEA_all_unique_countries_df = IEA_all_countries_df["countries"].unique()
# turning it into a series such that I can use Series.str.contains()
IEA_all_unique_countries_series = pd.Series(IEA_all_unique_countries_df.squeeze())
print(IEA_all_unique_countries_series)

0           [{"iso3":"USA","name":"United States"}]
1                  [{"iso3":"SRB","name":"Serbia"}]
2                 [{"iso3":"IRL","name":"Ireland"}]
3                 [{"iso3":"AUT","name":"Austria"}]
4            [{"iso3":"ZAF","name":"South Africa"}]
                           ...                     
386               [{"iso3":"NGA","name":"Nigeria"}]
387            [{"iso3":"MOZ","name":"Mozambique"}]
388              [{"iso3":"PAK","name":"Pakistan"}]
389              [{"iso3":"VNM","name":"Viet Nam"}]
390    [{"iso3":"RUS","name":"Russian Federation"}]
Length: 391, dtype: object


In [9]:
# creating a dataphrame of the country data to create the country filter from
policy_countries = pd.Series(policies_df['countries'].squeeze())
policy_countries.head()

0    [{"iso3":"USA","name":"United States"}]
1          [{"iso3":"IRL","name":"Ireland"}]
2     [{"iso3":"ZAF","name":"South Africa"}]
3    [{"iso3":"USA","name":"United States"}]
4            [{"iso3":"KOR","name":"Korea"}]
Name: countries, dtype: object

In [10]:
# creating an empty array for all of the countries we have full data for so I only have to cleen what we need
full_countries_binary = np.zeros(77)
# using a for loop to see which countries we have policy data for
i = 0
for country in country_full:
    if np.any(IEA_all_unique_countries_series.str.contains(country)):
        full_countries_binary[i] = True
    else:
        full_countries_binary[i] = False
    i += 1
# transforming the binary into a boolean
full_countries_binary = full_countries_binary == 1
full_countries_binary
# we have policy data for all the countries we have low carbon share data for so we don't have to remove any with this boolian array

array([ True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True])

In [11]:
# checking what the policy classifiers are
policies_df['status'].unique()

array(['In force', 'Ended', 'Announced', 'Achieved'], dtype=object)

In [12]:
low_carbon_spending_df['status'].unique()

array(['Ended', 'In force', 'Announced'], dtype=object)

In [13]:
carbon_pricing_df['status'].unique()

array(['In force', 'Ended', 'Announced'], dtype=object)

In [14]:
low_carbon_spending_ended = low_carbon_spending_df[low_carbon_spending_df['status'] == 'Ended']
low_carbon_spending_ended_null = low_carbon_spending_ended[low_carbon_spending_ended['yearEnded'].isnull()]
len(low_carbon_spending_ended_null)

15

In [1]:
# checking for other null data I need to clean
print(len(policies_df[policies_df['status'].isnull()]))
print(len(policies_df[policies_df['year'].isnull()]))
# classifying ended policies because a NaN year ended only matters for these data
ended_policies = policies_df[policies_df['status'] == 'Ended']
null_ended_policies = ended_policies[ended_policies['yearEnded'].isnull()]

# only nulls are in the year ended catagory which we can deal with in the counting process

NameError: name 'policies_df' is not defined

In [26]:
# creating an empty array for the policy count data
policy_counts = np.zeros(len(db2))
# using two for loops to fill out the data
i = 0
for country in country_full:
    for Year in range(1990, 2024):
        country_filter = policy_countries.str.contains(country)
        country_filtered = policies_df[country_filter]
        year_after_filtered = country_filtered[country_filtered['year'] <= Year]
        achieved_policies = len(year_after_filtered[year_after_filtered['status'] ==  'Achieved'])
        
        continued_policies = len(year_after_filtered[year_after_filtered['status'] == 'In force'])

        ended_filtered = year_after_filtered[year_after_filtered['status'] == 'Ended']
        clean_ended_policies = len(ended_filtered[ended_filtered['yearEnded'] <= Year])
        # from the data all we know about these policies is that they were in effect the year they were introduced thus that's the only year we'll count them for
        NaN_ended_policies = len(ended_filtered[ended_filtered['year'] == Year])
        
        policy_counts[i] = achieved_policies + continued_policies + clean_ended_policies + NaN_ended_policies
        i += 1
        
       



policy_counts

array([0., 0., 1., ..., 0., 0., 0.], shape=(2695,))

In [27]:
# adding data to the dataframe
db2["policy_count"] = policy_counts
db2.sort_values(by = ['policy_count'],ascending = False)

,Entity,Code,Year,Share of energy from low-carbon sources,GDP per capita,World region according to OWID,policy_count
10122,United Arab Emirates,ARE,2019,0.743936,72822.9700,Asia,27.0
10121,United Arab Emirates,ARE,2018,0.263157,72671.0600,Asia,27.0
650,Australia,AUS,2021,12.282586,58327.4140,Oceania,16.0
649,Australia,AUS,2020,10.292230,57260.4650,Oceania,15.0
10120,United Arab Emirates,ARE,2017,0.153722,72529.0550,Asia,15.0
...,...,...,...,...,...,...,...
10549,Vietnam,VNM,2015,17.797033,9248.0240,Asia,0.0
10550,Vietnam,VNM,2016,18.652882,9743.1880,Asia,0.0
10551,Vietnam,VNM,2017,23.116835,10290.5490,Asia,0.0
4336,Iraq,IRQ,1993,5.430123,6105.0415,Asia,0.0


In [39]:
db2.to_csv("PolicyData.csv", index=False)